# Feature Engineering

This notebook performs feature engineering for the credit card fraud detection project.
The features of the raw dataset are principal components obtained by applying PCA to the original features. Neither the original features nor information about them was provided. Therefore, feature engineering transformations will be minimal. Only the correlation coefficients between features will be calculated, aiming to remove highly correlated features.

## Objectives

The main goals of this notebook are:

* Calculate correlation coefficients between features;
* Generate a heatmap of the correlation coeffcients;
* Generate a correlation metadata dataframe. It contains the following information for each pair of features in the dataset: the method used to calculate the correlation, the p-value, the type of each feature, whether the features are normal, and the number of samples;
* Identify highly correlated features;
* Remove highly correlated features based on a specified threshold.


## Data Source

* Before running this notebook, the notebook 2-data_cleaning_and_split.ipynb must have been previously run. Otherwise, the train and test datasets won’t exist.

The training and testing dataset are located at:

- ..data/splits/train.parquet
- ..data/splits/test.parquet

Relative path considering that the notebook and the datasets are in different folders in the project root.

## Processing Steps

The following operations are performed in this notebook:

1. Load the train and test dataset
2. Calculate correlation between features;
3. Generate correlation heatmap
4. Generate correlation metadata dataframe
5. Remove highly correlated features

The resulting datasets are saved to:

- ..data/splits/feature_engineered/train_fe.parquet
- ..data/splits/feature_engineered/test_fe.parquet

Relative paths considering that the notebook and the datasets are in different folders in the project root.

## Notes

This notebook was used during the **research phase** of the project to prototype data preparation steps.

The final implementation of the data preparation workflow is included in the **production pipeline** located in the `src/datapipeline` package.


In [ ]:
import pandas as pd
import seaborn as sns
from pathlib import Path
import scipy as sp
import numpy as np
from typing import List
from scipy.stats import (skew, 
                        kurtosis, 
                        shapiro, 
                        pearsonr, 
                        normaltest, 
                        norm, 
                        spearmanr,
                        pointbiserialr,
                        kendalltau)
#import matplotlib.pyplot as plt
from scipy.stats.contingency import association

from pandas.api.types import (
    is_numeric_dtype,
    is_bool_dtype,
    is_datetime64_any_dtype,
    is_categorical_dtype,
    is_object_dtype,
    is_integer_dtype
    
)

# Config

In [ ]:
train_dataset_path = '../data/splits/train.parquet'
test_dataset_path = '../data/splits/test.parquet'
alpha = 0.05 #Significance level used for normality tests.
max_discrete_unique = 10 # Maximum number of unique values for a numeric feature to be considered discrete.
threshold_correlation = 0.95
remove_correlated = True
target_column = 'Class'
feature_engineered_folder_path = '../data/splits/feature_engineered'
train_feature_engineered_file = 'train_fe.parquet'
test_feature_engineered_file = 'test_fe.parquet'

# Load Data

In [ ]:
train_dataset_path = Path(train_dataset_path).resolve()
test_dataset_path = Path(test_dataset_path).resolve()

In [ ]:
df_train = pd.read_parquet(train_dataset_path)
df_test = pd.read_parquet(test_dataset_path)

In [ ]:
number_training_samples = len(df_train)
number_testing_samples = len(df_test)
total_number_samples = number_training_samples + number_testing_samples

print(f'{100*(number_training_samples/total_number_samples):.1f}% of samples for training')
print(f'{100*(number_testing_samples/total_number_samples):.1f}% of samples for training')

# Correlation

- A class was created to calculate the correlation coefficients between features. It has a general application; it can be applied to datasets containing every data type. It can be used to generate a correlation heatmap and a correlation metadata dataframe;
- Because the dataset has only numerical features, a simple function was created to calculate the correlation coefficients using the Pearson method.

In [ ]:
class FeatureCorrelation:
    """
    Performs an exploratory correlation analysis between features of a pandas 
    DataFrame.

    This class automatically:
    - infers feature types (continuous, binary, ordinal, categorical, etc.)
    - selects an appropriate correlation method based on feature types
      and distributional assumptions
    - computes correlation coefficients and optional statistical metadata

    The class is intended for exploratory data analysis (EDA) and
    diagnostics, not for production feature selection pipelines.

    Parameters:
        df (pd.DataFrame): Input dataframe containing features only.
        alpha (float): Significance level used for normality tests.
        max_discrete_unique (int): Numeric feature with a number of unique values below max_discrete_unique
            are considered discrete.
        ordinal_map (dict, optional): Mapping for ordinal categorical 
        variables.
    """

    def __init__(self,
                  df: pd.DataFrame, 
                  alpha: float = 0.05,
                  max_discrete_unique: int = 10, 
                  ordinal_map: dict | None = None):
        """
        Initializes the FeatureCorrelation analyzer and caches feature types.
        """
        self.df = df
        self.alpha = alpha
        self.max_discrete_unique = max_discrete_unique
        self.ordinal_map = ordinal_map or {}
         
        # Cache detected feature types
        self.feature_types = {
            col: self.detect_type(self.df[col]) for col in self.df.columns
         }

    # --------------------
    # Feature type detection
    # --------------------
    def detect_type(self, series: pd.Series) -> str:
       """
       Infers the semantic/statistical type of a pandas Series based on
       dtype, cardinality, and optional ordinal mapping.

       """
       series = series.dropna()

       if series.empty:
           return 'unknown'
       
       #boolean
       if is_bool_dtype(series):
           return 'binary'
       
       #datetime
       if is_datetime64_any_dtype(series):
           return 'datetime'
       
       if is_numeric_dtype(series):
           unique_vals = series.unique()

           #binary
           if set(unique_vals) == {0,1}:
               return 'binary'

           #discrete numeric
           if (
               is_integer_dtype(series) 
               and len(unique_vals) <= self.max_discrete_unique
           ):
               return 'discrete_numeric'

           return 'continuous'
       
       if is_categorical_dtype(series) or is_object_dtype(series):
           
           if self.ordinal_map is not None:
               if set(series.unique()).issubset(set(self.ordinal_map.keys())):
                   return 'ordinal'
           
           return 'categorical'
       
       return 'unknown'
    
    # --------------------
    # Normality check
    # --------------------
    def check_normality(self, series: pd.Series) -> bool:
        """
        Evaluates whether a numeric series can be considered approximately 
        normal.

        The decision is based on:
            - sample size (Shapiro-Wilk for n < 5000, 
                D'Agostino-Pearson otherwise)
            - skewness and kurtosis thresholds
            - significance level defined by alpha


        Returns:
            bool: True if the distribution is considered approximately normal.
        """

        series = series.dropna()

        if len(series) < 8:
            return False

        skewness = skew(series)
        kurt = kurtosis(series, fisher=True)

        n_samples = len(series)

        if n_samples<5000:
            _, p_value = shapiro(series)
        else:
            _, p_value = normaltest(series)

        if (p_value < self.alpha or 
            abs(skewness) > 1 or 
            abs(kurt) > 1):
            return False
       
        return True
    
    # --------------------
    # Correlation method selection
    # --------------------
    def choose_correlation(self, 
                           col_x: pd.Series, 
                           col_y: pd.Series) -> str:
        """
        Selects an appropriate correlation method based on the inferred
        types and distributional properties of two features.

        Possible methods include Pearson, Spearman, Kendall's Tau,
        and Cramér's V.

        Returns:
            str: Name of the selected correlation method, or 'not_applicable'
            if no suitable method is available.
        """
        tx = self.detect_type(col_x)
        ty = self.detect_type(col_y)    

        if tx == ty == 'continuous':
            if (self.check_normality(col_x) 
                and self.check_normality(col_y)):
                return 'pearson'
            return 'spearman'
            
        if tx == ty == 'binary':
            return 'pearson'
        
        if {"binary", "continuous"} == {tx, ty}:
            return 'pearson' 
            
        if tx == ty == 'discrete_numeric':
            return 'spearman' 

        if tx == ty == 'categorical':
            return 'cramers_v'

        if tx == ty == 'ordinal':
            return 'kendall_tau_b'
        
        if {"ordinal", "continuous"} == {tx, ty}:
            return "spearman"

        return 'not_applicable'
    
    # ------------------------------------------------------------------
    # Correlation computation (numeric only)
    # ------------------------------------------------------------------
    def calculate_corelation(self,
                              col_x: pd.Series, 
                              col_y: pd.Series, 
                              ) -> float:
        
        """
        Computes the correlation coefficient and p-value between two features
        using an automatically selected correlation method.

        Returns:
            Tuple[float, float]: Correlation coefficient and p-value.
            For methods where a p-value is not defined, NaN is returned.
        """
       
        correlation_method = self.choose_correlation(col_x, col_y)

        if correlation_method == 'pearson':
            res = pearsonr(col_x, col_y)
            return res.statistic, res.pvalue
        if correlation_method == 'spearman':
            res = spearmanr(col_x, col_y)
            return res.statistic, res.pvalue
        if correlation_method == 'kendall_tau_b':
            res = kendalltau(col_x, col_y)
            return res.statistic, res.pvalue
        if correlation_method == 'cramers_v':
            contigency_table = pd.crosstab(col_x, col_y)
            return association(contigency_table), np.nan
        return np.nan, np. nan
    
    # ------------------------------------------------------------------
    # Correlation matrix (numeric only)
    # ------------------------------------------------------------------
    
    def correlation_matrix(self) -> pd.DataFrame:
        """
        Computes a symmetric correlation matrix for all features
        in the dataframe.
        """
        
        features = self.df.columns
        corr_df = pd.DataFrame(np.nan,
                                index=features,
                                  columns=features)
        
        for i, f1 in enumerate(features):
            for j, f2 in enumerate(features):
                if j < i:
                    corr_df.loc[f1, f2] = corr_df.loc[f2, f1]
                elif i == j:
                    corr_df.loc[f1, f2] = 1.0
                else:
                    corr, _ = self.calculate_corelation(
                        self.df[f1], self.df[f2]
                    )
                    corr_df.loc[f1, f2] = corr
                    corr_df.loc[f2, f1] = corr
        return corr_df
    
    # ------------------------------------------------------------------
    # Correlation + metadata
    # ------------------------------------------------------------------

    def calculate_correlation_with_metadata(
        self, 
        col_x: pd.Series,
        col_y: pd.Series,
    ) -> dict:

        """
        Computes the correlation between two features along with
        diagnostic metadata such as feature types, normality flags,
        selected method, and sample size.

        Returns:
            dict: Dictionary containing correlation statistics and metadata.
        """

        valid_data = pd.concat([col_x, col_y], axis = 1).dropna()
        x = valid_data.iloc[:, 0]
        y = valid_data.iloc[:, 1]

        tx = self.detect_type(x)
        ty = self.detect_type(y)

        normal_x = self.check_normality(x) if tx == 'continuous' else None
        normal_y = self.check_normality(y) if ty == 'continuous' else None

        correlation_method = self.choose_correlation(x, y)
        
        if correlation_method == 'pearson':
            corr, p_value = pearsonr(x, y)
        elif correlation_method == 'spearman':
            corr, p_value = spearmanr(x, y)
        elif correlation_method == 'kendall_tau_b':
            corr, p_value = kendalltau(x, y)
        elif correlation_method == 'cramers_v':
            contigency_table = pd.crosstab(x, y)
            corr = association(contigency_table)
            p_value = np.nan    
        else:
            corr = np.nan
            p_value = np.nan

        return {
            'method': correlation_method,
            'correlation': corr,
            'p_value': p_value,
            'type_x': tx,
            'type_y': ty,
            'normal_x': normal_x,
            'normal_y': normal_y,
            'n_samples': len(valid_data)
        }
        
    
    # ------------------------------------------------------------------
    # Correlation + metadata
    # ------------------------------------------------------------------
      
    def correlation_metadata_table(self) -> pd.DataFrame:
        """
        Generates a table containing correlation statistics and metadata
        for all unique feature pairs.

        Returns:
            pd.DataFrame: Long-format table with one row per feature pair.
        """
        records = []
        features = self.df.columns

        for i, f1 in enumerate(features):
            for j, f2 in enumerate(features):
                if j <= i:
                    continue

                result = self.calculate_correlation_with_metadata(
                    self.df[f1], self.df[f2]
                )

                records.append({
                    "feature_x": f1,
                    "feature_y": f2,
                    **result
                })

        return pd.DataFrame(records)
    

In [ ]:
corr = FeatureCorrelation(df = df_train,
                          alpha = alpha,
                          max_discrete_unique = max_discrete_unique)


In [ ]:
correlation_matrix = corr.correlation_matrix()

In [ ]:
correlation_matrix

## Heatmap

In [ ]:
sns.heatmap(correlation_matrix)

## Metadata

In [ ]:
correlation_metadata = corr.correlation_metadata_table()

In [ ]:
correlation_metadata

## Feature Engineering

In [ ]:
def find_correlated_features(
    df: pd.DataFrame,
    threshold: float
    ) -> List[str]:

    """
    Identifies highly correlated features based on an absolute
    Pearson correlation threshold.

    This function is intended for use in production pipelines and
    should be applied ONLY to the training dataset.

    Args:
        df (pd.DataFrame): Training dataset 
        threshold (float): correlation threshold above which one 
                        of the features will be removed
    
    Returns:
        List[str]: List of feature names to be removed
    """

    if df.empty:
        return []

    # Compute absolute correlation matrix
    corr_matrix = df.corr().abs()

    # Upper triangle mask (exclude self-correlation)
    upper_triangle = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )

    #Identify features to drop
    to_drop = [feature for feature in upper_triangle if
               np.any(upper_triangle[feature]>threshold)]

    return to_drop

In [ ]:
def apply_feature_engineering(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    target_column: str,
    remove_correlated: bool,
    correlation_threshold: float,
    correlated_features: pd.DataFrame = None
    ):
    """
    Removes correlated features based on an absolute Pearson 
        correlation threshold. For a pair of features, if they 
        are highly correlated, the first that appears in the 
        dataframe will be removed

    Args:
        train_df (pd.DataFrame): training dataset
        test_df (pd.DataFrame): testing dataset
        target_column (str): target column of the dataset
        remove_correlated (bool): whther to remove correlated features
        correlation_threshold (float): threshold above which one of 
            the features will be removed
        correlated_features (list): list of correlated features
       
    Returns:
        train_df_fe (pd.DataFrame): feature engineered training dataset
        test_df_fe (pd.DataFrame): feature engineered testing dataset
    """

    X_train = train_df.drop(columns = target_column)
    X_test = test_df.drop(columns = target_column)

    if remove_correlated:
        if correlated_features is None:
            correlated_features = find_correlated_features(X_train, correlation_threshold)
        X_train = X_train.drop(columns = correlated_features)
        X_test = X_test.drop(columns = correlated_features)

        print(f'Correlated features removed: {correlated_features}')

    train_df_fe = pd.concat([X_train, train_df[target_column]], axis=1)
    test_df_fe = pd.concat([X_test, test_df[target_column]], axis=1)
    
    return train_df_fe, test_df_fe

In [ ]:
highly_correlated_features = find_correlated_features(
    df = df_train,
    threshold = threshold_correlation
)

if len(highly_correlated_features) == 0:
    print("There isn't highly correlated features in the dataset")
else:
    print(f'{len(highly_correlated_features)} features will be removed')

In [ ]:
train_df_fe, test_df_fe = apply_feature_engineering(
        train_df = df_train,
        test_df = df_test,
        target_column = target_column,
        remove_correlated = remove_correlated,
        correlation_threshold = threshold_correlation
)

In [ ]:
feature_engineered_folder_path = Path(train_feature_engineered_dataset_path).resolve()
feature_engineered_folder_path

In [ ]:
train_df_fe.to_parquet(feature_engineered_folder_path / train_feature_engineered_file)
test_df_fe.to_parquet(feature_engineered_folder_path / test_feature_engineered_file)